# retro_pipeline — interactive visualisation

This notebook lets you explore the output of each pipeline stage
interactively: metrics distributions, thermodynamic stability,
ranked designs, and 3D structures.

**Before running:**
1. Make sure you have the viz extras installed:
   `pip install -e ".[viz]"` from the `retro_pipeline/` directory.
2. The pipeline must have been run at least once (`--dry-run` is fine
   for exploring the scaffolding, but real data will be more interesting).

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

# Adjust if your notebook lives elsewhere.
WORKSPACE = Path.cwd().parent / "workspace"
print(f"Workspace: {WORKSPACE.resolve()}")
assert WORKSPACE.is_dir(), f"{WORKSPACE} does not exist"

In [ ]:
# -- Load all CSVs

metrics_csv = WORKSPACE / "03_predictions" / "metrics.csv"
ddg_csv     = WORKSPACE / "04_thermodynamics" / "ddg.csv"
ranked_csv  = WORKSPACE / "ranked_designs.csv"
pareto_csv  = WORKSPACE / "pareto_front.csv"

df_metrics = pd.read_csv(metrics_csv) if metrics_csv.exists() else pd.DataFrame()
df_ddg     = pd.read_csv(ddg_csv) if ddg_csv.exists() else pd.DataFrame()
df_ranked  = pd.read_csv(ranked_csv) if ranked_csv.exists() else pd.DataFrame()
df_pareto  = pd.read_csv(pareto_csv) if pareto_csv.exists() else pd.DataFrame()

print(f"Metrics:    {len(df_metrics)} rows")
print(f"DeltaDeltaG: {len(df_ddg)} rows")
print(f"Ranked:     {len(df_ranked)} rows")
print(f"Pareto:     {len(df_pareto)} rows")

---
## 1. Protenix — confidence metrics

Histograms and scatter plots showing how all designs scored on
pLDDT, ipTM, and interface PAE. Pass/fail are colour-coded.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")

if df_metrics.empty:
    print("No metrics data -- run Protenix stage first.")
else:
    fig, axes = plt.subplots(2, 2, figsize=(10, 8))

    passed = df_metrics[df_metrics["passed"] == True]
    failed = df_metrics[df_metrics["passed"] == False]

    plots = [
        ("plddt_designed_region_mean", "pLDDT (designed)", 0, 0),
        ("iptm", "ipTM", 0, 1),
        ("interface_pae_mean", "Interface PAE (A)", 1, 0),
        ("ranking_score", "Ranking score", 1, 1),
    ]
    for col, label, r, c in plots:
        if col not in df_metrics.columns:
            continue
        ax = axes[r, c]
        if not failed.empty:
            sns.histplot(failed[col], color="tomato", alpha=0.5, label="fail", ax=ax)
        sns.histplot(passed[col], color="steelblue", alpha=0.6, label="pass", ax=ax)
        ax.set_title(label)
        ax.set_ylabel("")
        if r == 0 and c == 1:
            ax.legend(fontsize=8)

    fig.suptitle("Protenix confidence metrics", fontsize=14, y=1.02)
    fig.tight_layout()
    plt.show()

In [ ]:
# Scatter: pLDDT vs ipTM (design-quality overview)
if not df_metrics.empty and all(c in df_metrics.columns for c in ("plddt_designed_region_mean", "iptm")):
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.scatterplot(
        data=df_metrics,
        x="plddt_designed_region_mean",
        y="iptm",
        hue="passed",
        alpha=0.6,
        ax=ax,
    )
    ax.set_title("pLDDT vs ipTM (each dot is one design)")
    plt.show()

---
## 2. FoldX — thermodynamic stability

DeltaDeltaG distribution and correlation with hydrophobic SASA.
Designs with DeltaDeltaG <= 0 are predicted to be at least as stable as wild-type.

In [ ]:
if df_ddg.empty:
    print("No DeltaDeltaG data -- run FoldX stage first.")
else:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

    # DeltaDeltaG distribution
    hue_col = "passed" if "passed" in df_ddg.columns else None
    sns.histplot(df_ddg, x="ddg_kcal_mol", hue=hue_col, ax=ax1)
    ax1.axvline(0, color="red", ls="--", lw=1, label="DeltaDeltaG = 0")
    ax1.set_title("DeltaDeltaG (kcal/mol)")
    ax1.legend()

    # DeltaDeltaG vs hydrophobic SASA
    if "hydrophobic_sasa" in df_ddg.columns:
        sns.scatterplot(
            df_ddg, x="ddg_kcal_mol", y="hydrophobic_sasa",
            hue=hue_col, alpha=0.6, ax=ax2,
        )
        ax2.axvline(0, color="red", ls="--", lw=1)
    ax2.set_title("DeltaDeltaG vs hydrophobic SASA")

    fig.suptitle("FoldX thermodynamic stability", fontsize=13, y=1.05)
    fig.tight_layout()
    plt.show()

---
## 3. Ranking & Pareto front

The final composite ranking, highlighting designs on the Pareto front.

In [ ]:
if df_ranked.empty:
    print("No ranking data -- run score_and_rank stage first.")
else:
    # Colour by composite score
    fig, ax = plt.subplots(figsize=(8, 6))
    sc = ax.scatter(
        df_ranked["plddt_designed"],
        df_ranked["iptm"],
        c=df_ranked["ddg_kcal_mol"],
        cmap="viridis_r",
        alpha=0.7,
        s=25,
    )
    if not df_pareto.empty:
        ax.scatter(
            df_pareto["plddt_designed"],
            df_pareto["iptm"],
            marker="D",
            facecolors="none",
            edgecolors="red",
            s=100,
            label="Pareto front",
        )
    ax.set_xlabel("pLDDT (designed)")
    ax.set_ylabel("ipTM")
    ax.set_title("Design ranking -- pLDDT vs ipTM (color = DeltaDeltaG)")
    plt.colorbar(sc, ax=ax, label="DeltaDeltaG (kcal/mol)")
    ax.legend()
    plt.show()

In [ ]:
import plotly.express as px
import plotly.graph_objects as go

if not df_ranked.empty:
    # Parallel coordinates -- great for multi-objective design space
    dims = ["ddg_kcal_mol", "iptm", "plddt_designed", "interface_pae", "hydrophobic_sasa"]
    dims = [d for d in dims if d in df_ranked.columns]

    fig = px.parallel_coordinates(
        df_ranked,
        dimensions=dims,
        color="composite_score" if "composite_score" in df_ranked.columns else dims[0],
        color_continuous_scale="viridis_r",
        title="Design ranking -- parallel coordinates",
    )
    fig.show()

In [ ]:
# 3D scatter of the objective space
if not df_ranked.empty and all(c in df_ranked.columns for c in ("ddg_kcal_mol", "iptm", "plddt_designed")):
    pareto_ids = set(df_pareto["job_name"]) if not df_pareto.empty else set()
    pareto_df = df_ranked[df_ranked["job_name"].isin(pareto_ids)] if "job_name" in df_ranked.columns else pd.DataFrame()

    fig = go.Figure()
    fig.add_trace(go.Scatter3d(
        x=df_ranked["ddg_kcal_mol"],
        y=df_ranked["iptm"],
        z=df_ranked["plddt_designed"],
        mode="markers",
        marker=dict(size=3, color=df_ranked["composite_score"], colorscale="Viridis_r", showscale=True),
        text=df_ranked.get("job_name", df_ranked.index),
        name="All designs",
    ))
    if not pareto_df.empty:
        fig.add_trace(go.Scatter3d(
            x=pareto_df["ddg_kcal_mol"],
            y=pareto_df["iptm"],
            z=pareto_df["plddt_designed"],
            mode="markers",
            marker=dict(size=8, color="red", symbol="diamond"),
            text=pareto_df["job_name"],
            name="Pareto front",
        ))
    fig.update_layout(
        height=600,
        title="3D objective space -- Pareto front highlighted",
        scene=dict(
            xaxis_title="DeltaDeltaG (kcal/mol)",
            yaxis_title="ipTM",
            zaxis_title="pLDDT",
        ),
    )
    fig.show()

---
## 4. 3D structure viewer

Inspect individual backbone PDBs or predicted CIF structures.
Each cell below will render an interactive 3D viewer inside this notebook.

In [ ]:
# Overlay the first few backbones to see structural diversity
from pathlib import Path

bb_dir = WORKSPACE / "01_backbones"
pdb_files = sorted(bb_dir.glob("*.pdb"))
print(f"Found {len(pdb_files)} backbone PDBs")
print(f"Showing first {min(5, len(pdb_files))}:")

import py3Dmol

if pdb_files:
    view = py3Dmol.view(width=800, height=500)
    for pdb in pdb_files[:5]:
        view.addModel(open(pdb).read(), "pdb")
    view.setStyle({"cartoon": {"color": "spectrum"}})
    view.zoomTo()
    view.show()

In [ ]:
# View predicted structures (passed designs)
passed_dir = WORKSPACE / "03_predictions" / "passed"
cif_files = sorted(passed_dir.glob("*.cif"))
print(f"Found {len(cif_files)} passed CIF structures")

import py3Dmol

if cif_files:
    view = py3Dmol.view(width=800, height=500)
    view.addModel(open(cif_files[0]).read(), "cif")
    view.setStyle({"cartoon": {"color": "spectrum"}})
    view.addSurface(py3Dmol.VDW, {"opacity": 0.3})
    view.zoomTo()
    view.show()

In [ ]:
# Browse a specific design by name
design_name = "sox2_00002__seq000"  # change to whichever design you want

cif_candidates = list(passed_dir.glob(f"{design_name}.cif")) if passed_dir.is_dir() else []
if cif_candidates:
    import py3Dmol
    view = py3Dmol.view(width=800, height=500)
    view.addModel(open(cif_candidates[0]).read(), "cif")
    view.setStyle({"cartoon": {"color": "spectrum"}})
    view.addLabel(design_name, {"fontSize": 14, "fontColor": "white", "backgroundColor": "black"})
    view.zoomTo()
    view.show()
else:
    print(f"Structure '{design_name}' not found in passed/")
    if cif_files:
        print(f"Available: {[p.stem for p in cif_files[:10]]}...")

---
## 5. Design table -- sort & search

Filter and sort the ranked design table interactively.

In [ ]:
# Display the full ranked table
if not df_ranked.empty:
    idx_col = "rank" if "rank" in df_ranked.columns else df_ranked.columns[0]
    display_df = df_ranked.set_index(idx_col)
    display(display_df.style.background_gradient(cmap="viridis_r", subset=["composite_score"]))
else:
    print("No ranking data.")

In [ ]:
# Pareto front table
if not df_pareto.empty:
    display(df_pareto.style.background_gradient(cmap="Blues"))
else:
    print("No Pareto front data.")

---
## Summary

| Plot | Data source | What to look for |
|---|---|---|
| pLDDT histogram | `metrics.csv` | Most designs >= 80? => high-confidence predictions |
| ipTM histogram | `metrics.csv` | ipTM > 0.6 => good binding confidence |
| Interface PAE | `metrics.csv` | Most < 10 A => good docking to DNA |
| DeltaDeltaG distribution | `ddg.csv` | Negative = more stable than WT |
| Pareto front | `ranked_designs.csv` | Non-dominated designs -- best tradeoffs |
| 3D structures | `passed/*.cif` | Visual inspection of top designs |